# Human Attention Forecasting in the Age of AI Content

**Sarfraz Hussain**  
Generative AI and Strategic Forecasting · Final Project

This project forecasts public attention across digital platforms and content categories for the next six months.

It uses Google Trends, Wikipedia page views, and Reddit community engagement data. The models used are ARIMA, Prophet, multivariate LSTM, and Temporal Fusion Transformer.

## Project scope

The approved idea promised a forecasting system for platforms and content categories.

The platform scope includes YouTube, TikTok, Instagram, LinkedIn, and Reddit.

The content scope includes ten categories. These are Artificial Intelligence and Technology, Personal Finance, Mental Health and Self Improvement, Sports, Gaming, News and Politics, Entertainment and Media, Science and Education, Lifestyle and Advice, and Culture Food and Travel.

The notebook separates model evaluation from the real future forecast. Backtesting checks how well the models predict known values. The future forecast predicts the next 26 weeks after the latest available data.

In [2]:
!pip install -q --force-reinstall "urllib3==1.26.18" "requests==2.31.0"
!pip install -q pytrends prophet neuralforecast groq gradio plotly statsmodels scikit-learn tensorflow streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.1/134.1 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.31.0 which is incompatible.
libpysal 4.14.1 requires requests>=2.32.0, but you have requests 2.31.0 which is incompatible.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.18 which is 

## Setup and configuration

All main settings are kept in one place. This makes the notebook easier to review and easier to change.

In [3]:
import os
import time
import random
import warnings
import zipfile
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.express as px

from pytrends.request import TrendReq

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA

from prophet import Prophet

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

try:
    from neuralforecast import NeuralForecast
    from neuralforecast.models import TFT
    TFT_AVAILABLE = True
    TFT_IMPORT_ERROR = ""
except Exception as e:
    TFT_AVAILABLE = False
    TFT_IMPORT_ERROR = str(e)

try:
    from groq import Groq
    GROQ_AVAILABLE = True
except Exception:
    GROQ_AVAILABLE = False

warnings.filterwarnings("ignore")

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_TITLE = "Human Attention Forecasting in the Age of AI Content"
STUDENT_ID = ""

PLATFORMS = ["YouTube", "TikTok", "Instagram", "LinkedIn", "Reddit"]

CONTENT_CATEGORIES = [
    "Artificial Intelligence and Technology",
    "Personal Finance",
    "Mental Health and Self Improvement",
    "Sports",
    "Gaming",
    "News and Politics",
    "Entertainment and Media",
    "Science and Education",
    "Lifestyle and Advice",
    "Culture Food and Travel"
]

CATEGORY_TREND_KEYWORDS = {
    "Artificial Intelligence and Technology": "Artificial Intelligence",
    "Personal Finance": "Personal Finance",
    "Mental Health and Self Improvement": "Mental Health",
    "Sports": "Sports",
    "Gaming": "Gaming",
    "News and Politics": "News",
    "Entertainment and Media": "Entertainment",
    "Science and Education": "Science",
    "Lifestyle and Advice": "Life Advice",
    "Culture Food and Travel": "Travel"
}

WIKIPEDIA_PAGES = {
    "YouTube": "YouTube",
    "TikTok": "TikTok",
    "Instagram": "Instagram",
    "LinkedIn": "LinkedIn",
    "Reddit": "Reddit",
    "Artificial Intelligence and Technology": "Artificial_intelligence",
    "Personal Finance": "Personal_finance",
    "Mental Health and Self Improvement": "Mental_health",
    "Sports": "Sport",
    "Gaming": "Video_game",
    "News and Politics": "News",
    "Entertainment and Media": "Entertainment",
    "Science and Education": "Science",
    "Lifestyle and Advice": "Advice_(opinion)",
    "Culture Food and Travel": "Travel"
}

SUBREDDIT_CATEGORY_MAP = {
    "technology": "Artificial Intelligence and Technology",
    "Futurology": "Artificial Intelligence and Technology",
    "personalfinance": "Personal Finance",
    "GetMotivated": "Mental Health and Self Improvement",
    "UpliftingNews": "Mental Health and Self Improvement",
    "wholesomememes": "Mental Health and Self Improvement",
    "sports": "Sports",
    "gaming": "Gaming",
    "news": "News and Politics",
    "worldnews": "News and Politics",
    "nottheonion": "News and Politics",
    "movies": "Entertainment and Media",
    "television": "Entertainment and Media",
    "Music": "Entertainment and Media",
    "anime": "Entertainment and Media",
    "videos": "Entertainment and Media",
    "funny": "Entertainment and Media",
    "memes": "Entertainment and Media",
    "Jokes": "Entertainment and Media",
    "gifs": "Entertainment and Media",
    "science": "Science and Education",
    "askscience": "Science and Education",
    "space": "Science and Education",
    "explainlikeimfive": "Science and Education",
    "todayilearned": "Science and Education",
    "history": "Science and Education",
    "Documentaries": "Science and Education",
    "AskReddit": "Lifestyle and Advice",
    "NoStupidQuestions": "Lifestyle and Advice",
    "LifeProTips": "Lifestyle and Advice",
    "relationship_advice": "Lifestyle and Advice",
    "IAmA": "Lifestyle and Advice",
    "Showerthoughts": "Lifestyle and Advice",
    "tifu": "Lifestyle and Advice",
    "food": "Culture Food and Travel",
    "travel": "Culture Food and Travel",
    "Art": "Culture Food and Travel",
    "DIY": "Culture Food and Travel",
    "EarthPorn": "Culture Food and Travel",
    "pics": "Culture Food and Travel",
    "InternetIsBeautiful": "Culture Food and Travel",
    "mildlyinteresting": "Culture Food and Travel",
    "interestingasfuck": "Culture Food and Travel",
    "OldSchoolCool": "Culture Food and Travel",
    "books": "Culture Food and Travel",
    "WritingPrompts": "Culture Food and Travel",
    "aww": "Culture Food and Travel",
    "creepy": "Culture Food and Travel",
    "dataisbeautiful": "Culture Food and Travel",
    "listentothis": "Culture Food and Travel"
}

END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")
START_DATE = (pd.Timestamp.today() - pd.DateOffset(years=3)).strftime("%Y-%m-%d")
TIMEFRAME = f"{START_DATE} {END_DATE}"
GEO = ""

BACKTEST_HORIZON = 26
FORECAST_HORIZON = 26
LSTM_LOOKBACK = 12
LSTM_EPOCHS = 35
LSTM_BATCH_SIZE = 32
TFT_INPUT_SIZE  = 52
TFT_HIDDEN_SIZE = 64
TFT_MAX_STEPS   = 300
USE_WIKIPEDIA = True
USE_GROQ = True
GROQ_API_KEY = ""

print(PROJECT_TITLE)
print("Student ID", STUDENT_ID)
print("Platforms", len(PLATFORMS))
print("Content categories", len(CONTENT_CATEGORIES))
print("Timeframe", TIMEFRAME)
print("GPU available", bool(tf.config.list_physical_devices("GPU")))
print("TFT available", TFT_AVAILABLE)
print("Groq available", GROQ_AVAILABLE)

Human Attention Forecasting in the Age of AI Content
Student ID 
Platforms 5
Content categories 10
Timeframe 2023-06-06 2026-06-06
GPU available True
TFT available True
Groq available True


## Data collection

This section collects the three promised attention signals.

Google Trends is the main public attention signal. Wikipedia page views are a supporting public interest signal. Reddit Kaggle data is a community engagement signal.

In [5]:
def normalize_0_100(series):
    series = pd.Series(series).astype(float)
    if series.dropna().empty:
        return series
    minimum = series.min()
    maximum = series.max()
    if maximum == minimum:
        return series * 0
    return ((series - minimum) / (maximum - minimum)) * 100


def clean_trends_data(df):
    df = df.copy()
    if "isPartial" in df.columns:
        df = df.drop(columns=["isPartial"])
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.interpolate(limit_direction="both")
    df = df.clip(0, 100)
    return df


def make_fallback_data(columns, periods=156):
    dates = pd.date_range(end=pd.Timestamp.today().normalize(), periods=periods, freq="W")
    values = {}
    for i, col in enumerate(columns):
        base = 25 + i * 3
        trend = np.linspace(0, np.random.uniform(-5, 12), periods)
        seasonality = 7 * np.sin(np.arange(periods) * 2 * np.pi / 52 + i)
        noise = np.random.normal(0, 3, periods)
        values[col] = np.clip(base + trend + seasonality + noise, 0, 100)
    return pd.DataFrame(values, index=dates)


def fetch_trends_group(keywords, retries=3, wait=8):
    last_error = None
    for attempt in range(retries):
        try:
            pytrends = TrendReq(hl="en-US", tz=0)
            pytrends.build_payload(
                kw_list=keywords,
                cat=0,
                timeframe=TIMEFRAME,
                geo=GEO,
                gprop=""
            )
            data = pytrends.interest_over_time()
            if data.empty:
                raise ValueError("Google Trends returned empty data")
            return clean_trends_data(data)
        except Exception as e:
            last_error = e
            time.sleep(wait * (attempt + 1))
    raise last_error

In [6]:
source_status = []

try:
    platform_trends = fetch_trends_group(PLATFORMS)
    platform_mode = "Live Google Trends"
except Exception as e:
    platform_trends = make_fallback_data(PLATFORMS)
    platform_mode = f"Fallback used after Google Trends failed with {e}"

source_status.append({"Source": "Google Trends platforms", "Status": platform_mode, "Rows": platform_trends.shape[0], "Columns": platform_trends.shape[1]})

print(platform_mode)
print(platform_trends.shape)
display(platform_trends.head())

Live Google Trends
(157, 5)


,YouTube,TikTok,Instagram,LinkedIn,Reddit
date,,,,,
2023-06-04,76,15,39,5,14
2023-06-11,74,16,38,6,14
2023-06-18,73,16,39,5,13
2023-06-25,74,16,39,5,14
2023-07-02,71,16,42,5,14


In [7]:
def fetch_category_trends(category_keywords):
    anchor_category = "Artificial Intelligence and Technology"
    other_categories = [cat for cat in category_keywords if cat != anchor_category]
    groups = []
    for i in range(0, len(other_categories), 4):
        group_categories = [anchor_category] + other_categories[i:i + 4]
        group_keywords = [category_keywords[cat] for cat in group_categories]
        groups.append((group_categories, group_keywords))
    group_frames = []
    for group_categories, group_keywords in groups:
        print("Fetching", group_keywords)
        raw = fetch_trends_group(group_keywords)
        raw.columns = group_categories
        group_frames.append(raw)
        time.sleep(8)
    base = group_frames[0].copy()
    base_anchor = base[anchor_category].replace(0, np.nan)
    final = pd.DataFrame(index=base.index)
    final[anchor_category] = base[anchor_category]
    for frame in group_frames:
        frame = frame.reindex(base.index).interpolate(limit_direction="both")
        frame_anchor = frame[anchor_category].replace(0, np.nan)
        ratio = (base_anchor / frame_anchor).replace([np.inf, -np.inf], np.nan).dropna()
        scale = ratio.median() if len(ratio) > 0 else 1
        for col in frame.columns:
            if col != anchor_category and col not in final.columns:
                final[col] = frame[col] * scale
    final = final[CONTENT_CATEGORIES]
    final = final.interpolate(limit_direction="both")
    final = final.clip(0, 100)
    return final

try:
    category_trends = fetch_category_trends(CATEGORY_TREND_KEYWORDS)
    category_mode = "Live Google Trends with anchor scaling"
except Exception as e:
    category_trends = make_fallback_data(CONTENT_CATEGORIES)
    category_mode = f"Fallback used after Google Trends failed with {e}"

source_status.append({"Source": "Google Trends content categories", "Status": category_mode, "Rows": category_trends.shape[0], "Columns": category_trends.shape[1]})

print(category_mode)
print(category_trends.shape)
display(category_trends.head())

Fetching ['Artificial Intelligence', 'Personal Finance', 'Mental Health', 'Sports', 'Gaming']
Fetching ['Artificial Intelligence', 'News', 'Entertainment', 'Science', 'Life Advice']
Fetching ['Artificial Intelligence', 'Travel']
Live Google Trends with anchor scaling
(157, 10)


,Artificial Intelligence and Technology,Personal Finance,Mental Health and Self Improvement,Sports,Gaming,News and Politics,Entertainment and Media,Science and Education,Lifestyle and Advice,Culture Food and Travel
date,,,,,,,,,,
2023-06-04,2,0.0,5.0,47.0,17.0,100.0,4.0,28.0,0.0,33.428571
2023-06-11,2,0.0,5.0,41.0,18.0,100.0,4.0,28.0,0.0,33.428571
2023-06-18,2,0.0,5.0,41.0,18.0,100.0,4.0,28.0,0.0,34.285714
2023-06-25,1,0.0,5.0,40.0,18.0,100.0,4.0,24.0,0.0,32.571429
2023-07-02,2,0.0,4.0,41.0,17.0,100.0,4.0,28.0,0.0,36.000000


## Wikipedia page views

Wikipedia page views are used as a supporting attention signal. They are not treated as the only source of truth.

In [8]:
def fetch_wikipedia_views(page):
    end_date = (datetime.utcnow().date() - timedelta(days=3)).strftime("%Y%m%d")
    start_date = (datetime.utcnow().date() - timedelta(days=365 * 3)).strftime("%Y%m%d")
    url = (
        "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
        f"en.wikipedia/all-access/all-agents/{page}/daily/{start_date}/{end_date}"
    )
    response = requests.get(url, headers={"User-Agent": "student-attention-forecasting-project"}, timeout=30)
    response.raise_for_status()
    items = response.json().get("items", [])
    if not items:
        raise ValueError("No page view records returned")
    df = pd.DataFrame(items)
    df["date"] = pd.to_datetime(df["timestamp"].str[:8], format="%Y%m%d")
    series = df.set_index("date")["views"].resample("W").sum()
    return normalize_0_100(series)

wiki_data = pd.DataFrame()
wiki_errors = []

if USE_WIKIPEDIA:
    for name, page in WIKIPEDIA_PAGES.items():
        try:
            wiki_data[name] = fetch_wikipedia_views(page)
            time.sleep(0.3)
        except Exception as e:
            wiki_errors.append({"Signal": name, "Error": str(e)})

wiki_data = wiki_data.sort_index().interpolate(limit_direction="both")

source_status.append({"Source": "Wikipedia page views", "Status": "Loaded with skipped pages if unavailable", "Rows": wiki_data.shape[0], "Columns": wiki_data.shape[1]})

print(wiki_data.shape)
display(wiki_data.head())
if wiki_errors:
    display(pd.DataFrame(wiki_errors))

(157, 10)


,YouTube,TikTok,Instagram,LinkedIn,Reddit,Artificial Intelligence and Technology,Personal Finance,Mental Health and Self Improvement,Sports,Gaming
date,,,,,,,,,,
2023-06-11,13.740825,5.843661,35.517974,5.043967,41.832469,8.015227,8.308458,8.402662,2.429176,40.127981
2023-06-18,24.608197,14.039290,77.806496,8.280222,100.000000,15.251574,30.845771,12.136635,2.924471,70.771816
2023-06-25,19.668450,9.725960,44.360497,6.930792,60.331874,14.579291,29.751244,13.453068,2.841778,67.086119
2023-07-02,8.298459,11.685994,65.029401,5.565867,38.416032,14.223034,24.029851,10.567355,2.504445,64.719551
2023-07-09,7.526106,11.842140,100.000000,5.757378,36.994315,13.745767,25.000000,11.722293,2.669314,60.043613


,Signal,Error
0,News and Politics,429 Client Error: Too Many Requests for url: h...
1,Entertainment and Media,429 Client Error: Too Many Requests for url: h...
2,Science and Education,429 Client Error: Too Many Requests for url: h...
3,Lifestyle and Advice,429 Client Error: Too Many Requests for url: h...
4,Culture Food and Travel,429 Client Error: Too Many Requests for url: h...


## Reddit Kaggle data

The Reddit dataset gives community engagement from fifty major subreddits. It is used as a Reddit engagement proxy, not as complete Reddit activity.

Each subreddit is mapped to one of the ten content categories. Weekly engagement is calculated from score and comments.

In [9]:
def find_reddit_zip():
    candidates = list(Path("/content").glob("*.zip")) + list(Path(".").glob("*.zip"))
    for path in candidates:
        if "archive" in path.name.lower() or "reddit" in path.name.lower():
            return path
    return candidates[0] if candidates else None

reddit_zip = find_reddit_zip()

if reddit_zip is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        reddit_zip = Path(list(uploaded.keys())[0])
    except Exception:
        reddit_zip = None

print("Reddit zip", reddit_zip)

Saving archive.zip to archive.zip
Reddit zip archive.zip


In [10]:
if reddit_zip is None:
    raise FileNotFoundError("Upload the Reddit Kaggle zip file before running this section")

extract_dir = Path("reddit_dataset")
extract_dir.mkdir(exist_ok=True)

with zipfile.ZipFile(reddit_zip, "r") as z:
    z.extractall(extract_dir)

csv_files = sorted(extract_dir.rglob("*.csv"))
frames = []

for file in csv_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    frames.append(temp)

reddit_raw = pd.concat(frames, ignore_index=True)
reddit_raw["created_utc"] = pd.to_datetime(reddit_raw["created_utc"], errors="coerce", utc=True)
reddit_raw = reddit_raw.dropna(subset=["created_utc", "subreddit"])

print("Rows", reddit_raw.shape[0])
print("Columns", reddit_raw.shape[1])
print("Date range", reddit_raw["created_utc"].min(), "to", reddit_raw["created_utc"].max())
print("Subreddits", reddit_raw["subreddit"].nunique())
display(reddit_raw.head())

Rows 48279
Columns 21
Date range 2011-09-27 04:17:32+00:00 to 2024-09-11 07:13:13+00:00
Subreddits 49


,subreddits,source_file,id,title,score,upvote_ratio,num_comments,created_utc,subreddit,subscribers,...,url,domain,num_awards,num_crossposts,crosspost_subreddits,post_type,is_nsfw,is_bot,is_megathread,body
50,NaN,anime.csv,1b9db8z,'Dragon Ball' Creator Akira Toryiyama Has Pass...,64352.0,0.95,3363.0,2024-03-08 08:31:22+00:00,anime,11104213.0,...,https://x.com/DB_official_en/status/1765935471...,x.com,0.0,24.0,NaN,link,False,False,False,NaN
51,NaN,anime.csv,jhqdgv,Kaguya-sama: Love Is War - Season 3 announced!,37983.0,0.98,1235.0,2020-10-25 14:30:48+00:00,anime,11104213.0,...,https://i.redd.it/7np2w8ngh7v51.png,i.redd.it,0.0,3.0,NaN,image,False,False,False,NaN
52,NaN,anime.csv,g9lda9,Aqua in yoga pants | Konosuba,31745.0,0.98,680.0,2020-04-28 16:45:45+00:00,anime,11104213.0,...,https://i.redd.it/5s7s3g2bljv41.png,i.redd.it,0.0,19.0,NaN,image,False,False,False,NaN
53,NaN,anime.csv,iqu61c,This is not a Cigarette [Gintama],31131.0,0.99,412.0,2020-09-11 22:13:34+00:00,anime,11104213.0,...,https://v.redd.it/v2eor03xrjm51,v.redd.it,0.0,10.0,NaN,video,False,False,False,NaN
54,NaN,anime.csv,lyzf3s,The Devil is a Part-Timer Season 2 Announced!,30752.0,0.99,2463.0,2021-03-06 16:37:09+00:00,anime,11104213.0,...,https://twitter.com/anime_maousama/status/1368...,twitter.com,0.0,5.0,NaN,link,False,False,False,NaN


In [11]:
reddit_data = reddit_raw.copy()
reddit_data["Category"] = reddit_data["subreddit"].map(SUBREDDIT_CATEGORY_MAP)
reddit_data = reddit_data.dropna(subset=["Category"])
reddit_data["score"] = pd.to_numeric(reddit_data["score"], errors="coerce").fillna(0)
reddit_data["num_comments"] = pd.to_numeric(reddit_data["num_comments"], errors="coerce").fillna(0)
reddit_data["engagement"] = reddit_data["score"] + reddit_data["num_comments"]
reddit_data["week"] = reddit_data["created_utc"].dt.tz_convert(None).dt.to_period("W").dt.start_time

reddit_weekly = (
    reddit_data
    .groupby(["week", "Category"])["engagement"]
    .sum()
    .unstack("Category")
    .sort_index()
)

reddit_weekly = reddit_weekly.reindex(columns=CONTENT_CATEGORIES).fillna(0)
reddit_weekly = reddit_weekly.apply(normalize_0_100)

reddit_category_summary = (
    reddit_data
    .groupby("Category")
    .agg(
        posts=("id", "count"),
        total_score=("score", "sum"),
        total_comments=("num_comments", "sum"),
        total_engagement=("engagement", "sum")
    )
    .reindex(CONTENT_CATEGORIES)
    .reset_index()
)

source_status.append({"Source": "Reddit Kaggle engagement", "Status": "Loaded and mapped to content categories", "Rows": reddit_weekly.shape[0], "Columns": reddit_weekly.shape[1]})

print(reddit_weekly.shape)
display(reddit_weekly.tail())
display(reddit_category_summary)

(548, 10)


Category,Artificial Intelligence and Technology,Personal Finance,Mental Health and Self Improvement,Sports,Gaming,News and Politics,Entertainment and Media,Science and Education,Lifestyle and Advice,Culture Food and Travel
week,,,,,,,,,,
2024-08-12,7.091967,2.722707,6.728881,5.196293,0.000000,5.784567,2.484989,2.343178,4.555696,6.027067
2024-08-19,4.784532,2.939508,8.484313,0.000000,0.000000,5.403098,7.233666,0.154529,3.935635,20.336995
2024-08-26,7.944809,4.263175,0.000000,4.661222,0.000000,1.640133,6.614340,2.100090,1.857355,13.136514
2024-09-02,4.308232,2.002862,2.376926,0.000000,4.545703,6.381677,5.421335,2.766850,2.109241,15.895309
2024-09-09,0.000000,1.780981,0.000000,0.000000,0.000000,0.000000,1.180684,0.000000,0.000000,8.049155


,Category,posts,total_score,total_comments,total_engagement
0,Artificial Intelligence and Technology,1990,88935378.0,5089949.0,94025327.0
1,Personal Finance,991,9258884.0,945120.0,10204004.0
2,Mental Health and Self Improvement,2922,162843302.0,2329760.0,165173062.0
3,Sports,985,43646143.0,1299958.0,44946101.0
4,Gaming,996,104477587.0,2226349.0,106703936.0
5,News and Politics,2971,227846851.0,13575778.0,241422629.0
6,Entertainment and Media,8937,588178582.0,17026060.0,605204642.0
7,Science and Education,6918,236370235.0,9446283.0,245816518.0
8,Lifestyle and Advice,5800,220899072.0,11798563.0,232697635.0
9,Culture Food and Travel,15769,714874863.0,17223857.0,732098720.0


## Source summary

This table shows which data sources were loaded and how they were used.

In [12]:
source_status_df = pd.DataFrame(source_status)
display(source_status_df)

,Source,Status,Rows,Columns
0,Google Trends platforms,Live Google Trends,157,5
1,Google Trends content categories,Live Google Trends with anchor scaling,157,10
2,Wikipedia page views,Loaded with skipped pages if unavailable,157,10
3,Reddit Kaggle engagement,Loaded and mapped to content categories,548,10


## Building the attention dataset

Platform attention uses Google Trends as the primary signal.

Content category attention combines Google Trends with available Wikipedia and Reddit category signals. Reddit is used where its Kaggle data overlaps with the project window. Google Trends keeps the category signal current for the final forecast.

In [13]:
def align_to_index(df, index, columns):
    aligned = df.copy()
    aligned.index = pd.to_datetime(aligned.index)
    aligned = aligned.reindex(index)
    aligned = aligned.reindex(columns=columns)
    aligned = aligned.interpolate(limit_direction="both")
    return aligned

platform_attention = platform_trends.copy()
category_google    = category_trends.copy()

wiki_platform = (
    align_to_index(wiki_data, platform_attention.index, PLATFORMS)
    if not wiki_data.empty
    else pd.DataFrame(index=platform_attention.index, columns=PLATFORMS)
)

wiki_category = (
    align_to_index(wiki_data, category_google.index, CONTENT_CATEGORIES)
    if not wiki_data.empty
    else pd.DataFrame(index=category_google.index, columns=CONTENT_CATEGORIES)
)

reddit_category = align_to_index(reddit_weekly, category_google.index, CONTENT_CATEGORIES)

category_attention = pd.DataFrame(index=category_google.index)

for category in CONTENT_CATEGORIES:
    sources = pd.concat(
        [
            category_google[category],
            reddit_category[category]
            if category in reddit_category.columns
            else pd.Series(index=category_google.index, dtype=float)
        ],
        axis=1
    )
    category_attention[category] = sources.mean(axis=1)

category_attention = category_attention.interpolate(limit_direction="both").clip(0, 100)

attention_data = pd.concat([platform_attention, category_attention], axis=1)
attention_data = attention_data.sort_index().interpolate(limit_direction="both").dropna()
attention_data = attention_data.tail(156)

PLATFORM_TARGETS  = PLATFORMS
CATEGORY_TARGETS  = CONTENT_CATEGORIES
ALL_TARGETS       = PLATFORM_TARGETS + CATEGORY_TARGETS

print("Training data sources: Google Trends (primary) + Reddit (secondary)")
print("Wikipedia is used in EDA only and is excluded from all model training.")
print("Platform attention :", platform_attention.shape)
print("Category attention :", category_attention.shape)
print("Final modeling data:", attention_data.shape)
display(attention_data.head())
display(attention_data.tail())

Training data sources: Google Trends (primary) + Reddit (secondary)
Wikipedia is used in EDA only and is excluded from all model training.
Platform attention : (157, 5)
Category attention : (157, 10)
Final modeling data: (156, 15)


,YouTube,TikTok,Instagram,LinkedIn,Reddit,Artificial Intelligence and Technology,Personal Finance,Mental Health and Self Improvement,Sports,Gaming,News and Politics,Entertainment and Media,Science and Education,Lifestyle and Advice,Culture Food and Travel
date,,,,,,,,,,,,,,,
2023-06-11,74,16,38,6,14,2.0,0.0,5.0,41.0,18.0,100.0,4.0,28.0,0.0,33.428571
2023-06-18,73,16,39,5,13,2.0,0.0,5.0,41.0,18.0,100.0,4.0,28.0,0.0,34.285714
2023-06-25,74,16,39,5,14,1.0,0.0,5.0,40.0,18.0,100.0,4.0,24.0,0.0,32.571429
2023-07-02,71,16,42,5,14,2.0,0.0,4.0,41.0,17.0,100.0,4.0,28.0,0.0,36.000000
2023-07-09,74,16,40,6,15,2.0,0.0,4.0,43.0,20.0,100.0,4.0,24.0,0.0,34.285714


,YouTube,TikTok,Instagram,LinkedIn,Reddit,Artificial Intelligence and Technology,Personal Finance,Mental Health and Self Improvement,Sports,Gaming,News and Politics,Entertainment and Media,Science and Education,Lifestyle and Advice,Culture Food and Travel
date,,,,,,,,,,,,,,,
2026-05-03,82,19,33,16,22,6.0,1.0,16.0,75.0,61.0,100.0,8.0,52.0,0.0,82.285714
2026-05-10,85,20,33,16,25,7.0,1.0,18.0,78.0,56.0,100.0,8.0,52.0,0.0,80.571429
2026-05-17,86,19,31,9,24,6.0,1.0,13.0,61.0,41.0,100.0,8.0,44.0,0.0,61.714286
2026-05-24,84,17,28,8,21,3.0,0.0,7.0,49.0,27.0,100.0,4.0,28.0,0.0,43.714286
2026-05-31,84,17,28,9,23,3.0,0.0,8.0,46.0,28.0,100.0,4.0,32.0,0.0,46.285714


## Exploratory analysis

The next charts show platform attention, content category attention, and Reddit engagement by category.

In [14]:
fig = go.Figure()
for col in PLATFORM_TARGETS:
    fig.add_trace(go.Scatter(x=attention_data.index, y=attention_data[col], mode="lines", name=col))
fig.update_layout(title="Platform attention over time", xaxis_title="Date", yaxis_title="Attention score", template="plotly_white", hovermode="x unified", height=500)
fig.show()

In [15]:
fig = go.Figure()
for col in CATEGORY_TARGETS:
    fig.add_trace(go.Scatter(x=attention_data.index, y=attention_data[col], mode="lines", name=col))
fig.update_layout(title="Content category attention over time", xaxis_title="Date", yaxis_title="Attention score", template="plotly_white", hovermode="x unified", height=600)
fig.show()

In [16]:
fig = go.Figure()
for col in CONTENT_CATEGORIES:
    fig.add_trace(go.Scatter(x=reddit_weekly.index, y=reddit_weekly[col], mode="lines", name=col))
fig.update_layout(title="Reddit engagement signal by content category", xaxis_title="Date", yaxis_title="Normalized Reddit engagement", template="plotly_white", hovermode="x unified", height=600)
fig.show()

In [17]:
latest_platforms = attention_data[PLATFORM_TARGETS].iloc[-1].sort_values(ascending=False)
latest_categories = attention_data[CATEGORY_TARGETS].iloc[-1].sort_values(ascending=False)

latest_summary = pd.concat([
    latest_platforms.rename("Latest score").to_frame().assign(Group="Platform"),
    latest_categories.rename("Latest score").to_frame().assign(Group="Content category")
])

latest_summary = latest_summary.reset_index().rename(columns={"index": "Signal"})
display(latest_summary)

,Signal,Latest score,Group
0,YouTube,84.000000,Platform
1,Instagram,28.000000,Platform
2,Reddit,23.000000,Platform
3,TikTok,17.000000,Platform
4,LinkedIn,9.000000,Platform
5,News and Politics,100.000000,Content category
6,Culture Food and Travel,46.285714,Content category
7,Sports,46.000000,Content category
8,Science and Education,32.000000,Content category
9,Gaming,28.000000,Content category


In [18]:
corr = attention_data.corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Correlation between attention signals")
fig.update_layout(template="plotly_white", height=850)
fig.show()

## Stationarity check

The stationarity test helps explain why differencing can be useful for ARIMA. It is used here as a diagnostic step before modeling.

In [19]:
stationarity_rows = []

for target in ALL_TARGETS:
    series = attention_data[target].dropna()
    try:
        result = adfuller(series)
        stationarity_rows.append({
            "Signal": target,
            "ADF statistic": result[0],
            "p value": result[1],
            "Stationary at 5 percent": result[1] < 0.05
        })
    except Exception:
        stationarity_rows.append({
            "Signal": target,
            "ADF statistic": np.nan,
            "p value": np.nan,
            "Stationary at 5 percent": False
        })

stationarity_df = pd.DataFrame(stationarity_rows)
display(stationarity_df)

,Signal,ADF statistic,p value,Stationary at 5 percent
0,YouTube,-2.067620,0.257732,False
1,TikTok,-2.491048,0.117668,False
2,Instagram,-2.459596,0.125617,False
3,LinkedIn,-0.406306,0.909041,False
4,Reddit,-1.429693,0.567961,False
5,Artificial Intelligence and Technology,-1.127962,0.703793,False
6,Personal Finance,-1.593705,0.486755,False
7,Mental Health and Self Improvement,-1.022644,0.744916,False
8,Sports,-2.502645,0.114832,False
9,Gaming,-1.544760,0.511265,False


## Backtest setup

The last 26 weeks are kept as a historical backtest period. The models train on earlier data and predict this held out period.

This is different from the true future forecast, which predicts the next 26 weeks after the latest available data.

In [20]:
model_data = attention_data.copy().sort_index().interpolate(limit_direction="both").dropna()

train_data = model_data.iloc[:-BACKTEST_HORIZON].copy()
test_data = model_data.iloc[-BACKTEST_HORIZON:].copy()
train_dates = train_data.index
test_dates = test_data.index
future_dates = pd.date_range(start=model_data.index[-1] + pd.Timedelta(weeks=1), periods=FORECAST_HORIZON, freq="W")

backtest_results = []
backtest_predictions = {}
future_predictions = {}

print("Full data", model_data.shape)
print("Train data", train_data.shape)
print("Test data", test_data.shape)
print("Backtest starts", test_dates.min())
print("Backtest ends", test_dates.max())
print("Future forecast starts", future_dates.min())
print("Future forecast ends", future_dates.max())

Full data (156, 15)
Train data (130, 15)
Test data (26, 15)
Backtest starts 2025-12-07 00:00:00
Backtest ends 2026-05-31 00:00:00
Future forecast starts 2026-06-07 00:00:00
Future forecast ends 2026-11-29 00:00:00


In [21]:
def calculate_metrics(actual, predicted):
    actual = np.array(actual, dtype=float)
    predicted = np.clip(np.array(predicted, dtype=float), 0, 100)
    mask = actual != 0
    mape = np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100 if mask.sum() > 0 else np.nan
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE": mape
    }


def add_result(model_name, target, actual, predicted):
    metrics = calculate_metrics(actual, predicted)
    backtest_results.append({
        "Model": model_name,
        "Target": target,
        "Group": "Platform" if target in PLATFORM_TARGETS else "Content category",
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "MAPE": metrics["MAPE"]
    })
    backtest_predictions[(model_name, target)] = np.clip(np.array(predicted, dtype=float), 0, 100)

## ARIMA

ARIMA is used as the statistical baseline. Each platform and content category is modeled separately.

In [22]:
def find_best_arima_order(series):
    best_aic = np.inf
    best_order = None
    best_model = None
    for p in [0, 1, 2]:
        for d in [0, 1]:
            for q in [0, 1, 2]:
                if p == 0 and d == 0 and q == 0:
                    continue
                try:
                    fitted = ARIMA(series, order=(p, d, q)).fit()
                    if fitted.aic < best_aic:
                        best_aic = fitted.aic
                        best_order = (p, d, q)
                        best_model = fitted
                except Exception:
                    pass
    if best_model is None:
        best_order = (1, 1, 1)
        best_model = ARIMA(series, order=best_order).fit()
    return best_model, best_order

arima_orders = {}

for target in ALL_TARGETS:
    fitted, order = find_best_arima_order(train_data[target])
    forecast = np.clip(fitted.forecast(steps=BACKTEST_HORIZON), 0, 100)
    add_result("ARIMA", target, test_data[target].values, forecast)
    arima_orders[target] = order

arima_summary = pd.DataFrame([{"Target": target, "ARIMA order": arima_orders[target]} for target in ALL_TARGETS])
display(arima_summary)
display(pd.DataFrame(backtest_results).query("Model == 'ARIMA'").sort_values("MAE"))

,Target,ARIMA order
0,YouTube,"(1, 1, 1)"
1,TikTok,"(1, 1, 2)"
2,Instagram,"(0, 1, 1)"
3,LinkedIn,"(0, 1, 2)"
4,Reddit,"(0, 1, 2)"
5,Artificial Intelligence and Technology,"(0, 1, 1)"
6,Personal Finance,"(0, 1, 0)"
7,Mental Health and Self Improvement,"(2, 1, 2)"
8,Sports,"(0, 1, 1)"
9,Gaming,"(2, 0, 2)"


,Model,Target,Group,MAE,RMSE,MAPE
13,ARIMA,Lifestyle and Advice,Content category,0.000000,0.000000,NaN
10,ARIMA,News and Politics,Content category,0.000000,0.000000,0.000000
6,ARIMA,Personal Finance,Content category,0.076923,0.277350,0.000000
1,ARIMA,TikTok,Platform,0.926124,1.078920,5.267442
2,ARIMA,Instagram,Platform,1.092081,1.406428,3.622280
5,ARIMA,Artificial Intelligence and Technology,Content category,1.219289,1.480247,24.330483
11,ARIMA,Entertainment and Media,Content category,1.864067,1.871735,26.417843
7,ARIMA,Mental Health and Self Improvement,Content category,2.231944,2.827190,22.901375
3,ARIMA,LinkedIn,Platform,2.657805,3.440717,25.103968
4,ARIMA,Reddit,Platform,3.016226,4.321439,13.281497


## Prophet

Prophet is used to model trend and seasonality. It is also applied to both platforms and content categories.

In [23]:
event_columns = ["ai_content_shift", "platform_policy_attention", "creator_economy_shift"]


def add_event_regressors(df):
    df = df.copy()
    df["ai_content_shift"] = ((df["ds"] >= pd.Timestamp("2022-11-01")) & (df["ds"] <= pd.Timestamp("2023-05-31"))).astype(int)
    df["platform_policy_attention"] = ((df["ds"] >= pd.Timestamp("2024-03-01")) & (df["ds"] <= pd.Timestamp("2024-07-31"))).astype(int)
    df["creator_economy_shift"] = ((df["ds"] >= pd.Timestamp("2023-01-01")) & (df["ds"] <= pd.Timestamp("2023-12-31"))).astype(int)
    return df


def fit_prophet_forecast(series, steps):
    train_df = pd.DataFrame({"ds": series.index, "y": series.values})
    train_df = add_event_regressors(train_df)
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    for col in event_columns:
        model.add_regressor(col)
    model.fit(train_df)
    future = model.make_future_dataframe(periods=steps, freq="W")
    future = add_event_regressors(future)
    forecast = model.predict(future)
    return np.clip(forecast.tail(steps)["yhat"].values, 0, 100), model

prophet_models = {}

for target in ALL_TARGETS:
    forecast, fitted = fit_prophet_forecast(train_data[target], BACKTEST_HORIZON)
    add_result("Prophet", target, test_data[target].values, forecast)
    prophet_models[target] = fitted

display(pd.DataFrame(backtest_results).query("Model == 'Prophet'").sort_values("MAE"))

,Model,Target,Group,MAE,RMSE,MAPE
28,Prophet,Lifestyle and Advice,Content category,0.000000,0.000000,NaN
25,Prophet,News and Politics,Content category,0.000000,0.000000,0.000000
21,Prophet,Personal Finance,Content category,0.137585,0.297522,6.273616
20,Prophet,Artificial Intelligence and Technology,Content category,1.250844,1.579462,23.951076
16,Prophet,TikTok,Platform,1.426141,2.002165,8.349474
17,Prophet,Instagram,Platform,1.574196,1.866221,5.126874
22,Prophet,Mental Health and Self Improvement,Content category,1.733667,2.257445,16.738413
18,Prophet,LinkedIn,Platform,1.957992,2.813726,17.942871
26,Prophet,Entertainment and Media,Content category,2.102422,2.118914,28.688692
19,Prophet,Reddit,Platform,3.470660,4.610860,15.727887


## Multivariate LSTM

The LSTM uses all platform and content category signals together. It predicts all signals at the same time.

The scaler is fitted only on training data. During backtesting, the model forecasts recursively from the end of the training period.

In [24]:
feature_columns = model_data.columns.tolist()
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data[feature_columns])


def make_lstm_sequences(values, lookback):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i - lookback:i])
        y.append(values[i])
    return np.array(X), np.array(y)


def recursive_lstm_forecast(model, start_window, steps):
    window = start_window.copy()
    predictions = []
    for _ in range(steps):
        next_pred = model.predict(window.reshape(1, window.shape[0], window.shape[1]), verbose=0)[0]
        predictions.append(next_pred)
        window = np.vstack([window[1:], next_pred])
    return np.array(predictions)

X_train_lstm, y_train_lstm = make_lstm_sequences(train_scaled, LSTM_LOOKBACK)

lstm_model = Sequential([
    LSTM(64, input_shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])),
    Dropout(0.2),
    Dense(len(feature_columns))
])

lstm_model.compile(optimizer="adam", loss="mse")

history = lstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=LSTM_EPOCHS,
    batch_size=LSTM_BATCH_SIZE,
    validation_split=0.2,
    callbacks=[EarlyStopping(patience=6, restore_best_weights=True)],
    verbose=0
)

start_window = train_scaled[-LSTM_LOOKBACK:]
lstm_scaled_backtest = recursive_lstm_forecast(lstm_model, start_window, BACKTEST_HORIZON)
lstm_backtest = np.clip(scaler.inverse_transform(lstm_scaled_backtest), 0, 100)

for i, target in enumerate(feature_columns):
    add_result("LSTM", target, test_data[target].values, lstm_backtest[:, i])

print("LSTM training complete")
print("Final validation loss", history.history["val_loss"][-1])
display(pd.DataFrame(backtest_results).query("Model == 'LSTM'").sort_values("MAE"))

LSTM training complete
Final validation loss 0.15398356318473816


,Model,Target,Group,MAE,RMSE,MAPE
43,LSTM,Lifestyle and Advice,Content category,0.000000,0.000000,NaN
40,LSTM,News and Politics,Content category,0.042824,0.048147,0.042824
36,LSTM,Personal Finance,Content category,0.826876,0.855257,88.939539
31,LSTM,TikTok,Platform,1.026504,1.281613,5.643079
32,LSTM,Instagram,Platform,2.015118,2.397855,6.740543
30,LSTM,YouTube,Platform,2.612093,3.417681,2.910542
33,LSTM,LinkedIn,Platform,2.702493,3.500038,24.834983
35,LSTM,Artificial Intelligence and Technology,Content category,3.188790,3.448280,64.412136
34,LSTM,Reddit,Platform,3.744954,4.269579,19.565269
41,LSTM,Entertainment and Media,Content category,3.788505,3.993445,47.881061


## Temporal Fusion Transformer

TFT is included because it was part of the approved idea. It is trained on all platform and content category signals in long time series format.

In [25]:
tft_results_available = False
tft_error_message = ""

if TFT_AVAILABLE:
    try:
        tft_train_rows = []
        for target in ALL_TARGETS:
            temp = train_data[[target]].reset_index()
            temp.columns = ["ds", "y"]
            temp["unique_id"] = target
            tft_train_rows.append(temp[["unique_id", "ds", "y"]])
        tft_train_df = pd.concat(tft_train_rows, ignore_index=True)
        tft_model = TFT(h=BACKTEST_HORIZON, input_size=TFT_INPUT_SIZE, hidden_size=TFT_HIDDEN_SIZE, max_steps=TFT_MAX_STEPS, random_seed=SEED)
        tft_forecaster = NeuralForecast(models=[tft_model], freq="W")
        tft_forecaster.fit(tft_train_df)
        tft_forecast = tft_forecaster.predict()
        prediction_column = [col for col in tft_forecast.columns if col not in ["unique_id", "ds"]][0]
        for target in ALL_TARGETS:
            target_forecast = tft_forecast[tft_forecast["unique_id"] == target]
            predicted = np.clip(target_forecast[prediction_column].values[:BACKTEST_HORIZON], 0, 100)
            add_result("TFT", target, test_data[target].values, predicted)
        tft_results_available = True
    except Exception as e:
        tft_error_message = str(e)
else:
    tft_error_message = TFT_IMPORT_ERROR

if tft_results_available:
    display(pd.DataFrame(backtest_results).query("Model == 'TFT'").sort_values("MAE"))
else:
    print("TFT did not produce results")
    print(tft_error_message)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 256    | train
4 | temporal_encoder        | TemporalCovariateEncoder | 154 K  | train
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 64.8 K | train
6 | output_adapter          | Linear                   | 65     | tra

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

,Model,Target,Group,MAE,RMSE,MAPE
55,TFT,News and Politics,Content category,0.000209,0.001067,0.000209
58,TFT,Lifestyle and Advice,Content category,0.012538,0.012975,NaN
51,TFT,Personal Finance,Content category,0.197259,0.265576,14.829817
56,TFT,Entertainment and Media,Content category,0.716813,1.267719,14.018846
50,TFT,Artificial Intelligence and Technology,Content category,1.158941,1.445192,26.238885
46,TFT,TikTok,Platform,1.191679,1.415700,6.994360
47,TFT,Instagram,Platform,2.195812,2.787580,7.340123
52,TFT,Mental Health and Self Improvement,Content category,2.476633,3.120994,23.788277
48,TFT,LinkedIn,Platform,2.761637,3.533319,26.018291
49,TFT,Reddit,Platform,2.944551,3.898174,13.623215


## Model comparison

This section compares all models on the same 26 week backtest period.

In [26]:
results_df = pd.DataFrame(backtest_results)
results_df = results_df.drop_duplicates(subset=["Model", "Target"], keep="last")
results_df = results_df.sort_values(["Group", "Target", "MAE"])

display(results_df)

model_summary = (
    results_df
    .groupby(["Model", "Group"], as_index=False)
    .agg(Average_MAE=("MAE", "mean"), Average_RMSE=("RMSE", "mean"), Average_MAPE=("MAPE", "mean"))
    .sort_values(["Group", "Average_MAE"])
)

display(model_summary)

best_models = (
    results_df
    .sort_values("MAE")
    .groupby("Target", as_index=False)
    .first()
)

display(best_models)

,Model,Target,Group,MAE,RMSE,MAPE
50,TFT,Artificial Intelligence and Technology,Content category,1.158941,1.445192,26.238885
5,ARIMA,Artificial Intelligence and Technology,Content category,1.219289,1.480247,24.330483
20,Prophet,Artificial Intelligence and Technology,Content category,1.250844,1.579462,23.951076
35,LSTM,Artificial Intelligence and Technology,Content category,3.188790,3.448280,64.412136
29,Prophet,Culture Food and Travel,Content category,8.910151,11.770222,15.188240
59,TFT,Culture Food and Travel,Content category,9.835558,11.725938,14.949283
14,ARIMA,Culture Food and Travel,Content category,10.432381,11.864414,17.436625
44,LSTM,Culture Food and Travel,Content category,28.371655,30.372739,41.857442
56,TFT,Entertainment and Media,Content category,0.716813,1.267719,14.018846
11,ARIMA,Entertainment and Media,Content category,1.864067,1.871735,26.417843


,Model,Group,Average_MAE,Average_RMSE,Average_MAPE
6,TFT,Content category,3.766015,4.684551,16.276237
4,Prophet,Content category,3.856973,4.678842,16.504788
0,ARIMA,Content category,4.247488,5.184286,16.675584
2,LSTM,Content category,7.711530,8.855369,39.287522
3,LSTM,Platform,2.420232,2.973353,11.938883
1,ARIMA,Platform,3.107407,3.745940,11.273524
5,Prophet,Platform,3.836836,4.582269,11.922095
7,TFT,Platform,4.211044,4.866185,13.563127


,Target,Model,Group,MAE,RMSE,MAPE
0,Artificial Intelligence and Technology,TFT,Content category,1.158941,1.445192,26.238885
1,Culture Food and Travel,Prophet,Content category,8.910151,11.770222,15.188240
2,Entertainment and Media,TFT,Content category,0.716813,1.267719,14.018846
3,Gaming,TFT,Content category,9.813478,12.286814,24.109689
4,Instagram,ARIMA,Platform,1.092081,1.406428,3.622280
5,Lifestyle and Advice,LSTM,Content category,0.000000,0.000000,NaN
6,LinkedIn,Prophet,Platform,1.957992,2.813726,17.942871
7,Mental Health and Self Improvement,Prophet,Content category,1.733667,2.257445,16.738413
8,News and Politics,ARIMA,Content category,0.000000,0.000000,0.000000
9,Personal Finance,ARIMA,Content category,0.076923,0.277350,0.000000


In [27]:
fig = px.bar(
    model_summary,
    x="Model",
    y="Average_MAE",
    color="Group",
    barmode="group",
    title="Average backtest MAE by model and target group",
    text_auto=".2f"
)
fig.update_layout(template="plotly_white", height=500)
fig.show()

In [28]:
example_target = "YouTube"

fig = go.Figure()
fig.add_trace(go.Scatter(x=test_dates, y=test_data[example_target], mode="lines+markers", name="Actual"))

for model_name in results_df["Model"].unique():
    key = (model_name, example_target)
    if key in backtest_predictions:
        fig.add_trace(go.Scatter(x=test_dates, y=backtest_predictions[key], mode="lines+markers", name=model_name))

fig.update_layout(title="Backtest comparison for YouTube", xaxis_title="Date", yaxis_title="Attention score", template="plotly_white", hovermode="x unified", height=550)
fig.show()

## True future forecast

This section predicts the next 26 weeks after the latest available date. This is the main forecast output.

In [29]:
def arima_future(series, steps):
    fitted, order = find_best_arima_order(series)
    return np.clip(fitted.forecast(steps=steps), 0, 100)


def prophet_future(series, steps):
    forecast, fitted = fit_prophet_forecast(series, steps)
    return forecast

arima_future_predictions = {}
prophet_future_predictions = {}

for target in ALL_TARGETS:
    arima_future_predictions[target] = arima_future(model_data[target], FORECAST_HORIZON)
    prophet_future_predictions[target] = prophet_future(model_data[target], FORECAST_HORIZON)

full_scaler = MinMaxScaler()
full_scaled = full_scaler.fit_transform(model_data[feature_columns])
X_full_lstm, y_full_lstm = make_lstm_sequences(full_scaled, LSTM_LOOKBACK)

lstm_future_model = Sequential([
    LSTM(64, input_shape=(X_full_lstm.shape[1], X_full_lstm.shape[2])),
    Dropout(0.2),
    Dense(len(feature_columns))
])

lstm_future_model.compile(optimizer="adam", loss="mse")
lstm_future_model.fit(X_full_lstm, y_full_lstm, epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH_SIZE, validation_split=0.2, callbacks=[EarlyStopping(patience=6, restore_best_weights=True)], verbose=0)

lstm_future_scaled = recursive_lstm_forecast(lstm_future_model, full_scaled[-LSTM_LOOKBACK:], FORECAST_HORIZON)
lstm_future_values = np.clip(full_scaler.inverse_transform(lstm_future_scaled), 0, 100)
lstm_future_predictions = {target: lstm_future_values[:, i] for i, target in enumerate(feature_columns)}

print("Stable future models complete")

Stable future models complete


In [30]:
tft_future_available = False
tft_future_predictions = {}
tft_future_error = ""

if TFT_AVAILABLE:
    try:
        tft_full_rows = []
        for target in ALL_TARGETS:
            temp = model_data[[target]].reset_index()
            temp.columns = ["ds", "y"]
            temp["unique_id"] = target
            tft_full_rows.append(temp[["unique_id", "ds", "y"]])
        tft_full_df = pd.concat(tft_full_rows, ignore_index=True)
        tft_future_model = TFT(h=FORECAST_HORIZON, input_size=TFT_INPUT_SIZE, hidden_size=TFT_HIDDEN_SIZE, max_steps=TFT_MAX_STEPS, random_seed=SEED)
        tft_future_forecaster = NeuralForecast(models=[tft_future_model], freq="W")
        tft_future_forecaster.fit(tft_full_df)
        tft_future_df = tft_future_forecaster.predict()
        tft_prediction_column = [col for col in tft_future_df.columns if col not in ["unique_id", "ds"]][0]
        for target in ALL_TARGETS:
            temp = tft_future_df[tft_future_df["unique_id"] == target]
            tft_future_predictions[target] = np.clip(temp[tft_prediction_column].values[:FORECAST_HORIZON], 0, 100)
        tft_future_available = True
    except Exception as e:
        tft_future_error = str(e)

if tft_future_available:
    print("TFT future forecast complete")
else:
    print("TFT future forecast not available")
    print(tft_future_error)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                    | Type                     | Params | Mode 
-----------------------------------------------------------------------------
0 | loss                    | MAE                      | 0      | train
1 | padder_train            | ConstantPad1d            | 0      | train
2 | scaler                  | TemporalNorm             | 0      | train
3 | embedding               | TFTEmbedding             | 256    | train
4 | temporal_encoder        | TemporalCovariateEncoder | 154 K  | train
5 | temporal_fusion_decoder | TemporalFusionDecoder    | 64.8 K | train
6 | output_adapter          | Linear                   | 65     | tra

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=300` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting: |          | 0/? [00:00<?, ?it/s]

TFT future forecast complete


In [31]:
future_model_outputs = {
    "ARIMA": arima_future_predictions,
    "Prophet": prophet_future_predictions,
    "LSTM": lstm_future_predictions
}

if tft_future_available:
    future_model_outputs["TFT"] = tft_future_predictions

best_model_map = dict(zip(best_models["Target"], best_models["Model"]))
future_rows = []
summary_rows = []

for target in ALL_TARGETS:
    selected_model = best_model_map[target]
    if selected_model not in future_model_outputs:
        selected_model = "ARIMA"
    values = np.clip(future_model_outputs[selected_model][target], 0, 100)
    for date, value in zip(future_dates, values):
        future_rows.append({
            "Date": date,
            "Target": target,
            "Group": "Platform" if target in PLATFORM_TARGETS else "Content category",
            "Selected model": selected_model,
            "Forecast": value
        })
    start_value = values[0]
    end_value = values[-1]
    change = end_value - start_value
    percent_change = (change / start_value) * 100 if start_value != 0 else np.nan
    summary_rows.append({
        "Target": target,
        "Group": "Platform" if target in PLATFORM_TARGETS else "Content category",
        "Selected model": selected_model,
        "Forecast start": start_value,
        "Forecast end": end_value,
        "Expected change": change,
        "Expected percent change": percent_change
    })

future_forecasts = pd.DataFrame(future_rows)
future_summary = pd.DataFrame(summary_rows).sort_values(["Group", "Expected percent change"], ascending=[True, False])
platform_future_summary = future_summary[future_summary["Group"] == "Platform"].copy()
category_future_summary = future_summary[future_summary["Group"] == "Content category"].copy()

display(future_summary)

,Target,Group,Selected model,Forecast start,Forecast end,Expected change,Expected percent change
11,Entertainment and Media,Content category,TFT,7.173254e+00,1.511131e+01,7.938060,110.661911
5,Artificial Intelligence and Technology,Content category,TFT,3.576907e+00,6.669136e+00,3.092228,86.449768
12,Science and Education,Content category,Prophet,4.051126e+01,5.519334e+01,14.682076,36.241964
7,Mental Health and Self Improvement,Content category,Prophet,1.131392e+01,1.529055e+01,3.976635,35.148168
14,Culture Food and Travel,Content category,Prophet,6.879133e+01,9.069526e+01,21.903926,31.841112
8,Sports,Content category,ARIMA,4.845221e+01,5.619866e+01,7.746450,15.987816
9,Gaming,Content category,TFT,3.218643e+01,3.486806e+01,2.681625,8.331539
6,Personal Finance,Content category,ARIMA,1.232595e-32,1.232595e-32,0.000000,0.000000
10,News and Politics,Content category,ARIMA,1.000000e+02,1.000000e+02,0.000000,0.000000
13,Lifestyle and Advice,Content category,LSTM,0.000000e+00,0.000000e+00,0.000000,NaN


In [32]:
fig = go.Figure()
for target in PLATFORM_TARGETS:
    temp = future_forecasts[future_forecasts["Target"] == target]
    fig.add_trace(go.Scatter(x=temp["Date"], y=temp["Forecast"], mode="lines+markers", name=target))
fig.update_layout(title="True 26 week future forecast by platform", xaxis_title="Future date", yaxis_title="Forecasted attention score", template="plotly_white", hovermode="x unified", height=550)
fig.show()

In [33]:
fig = go.Figure()
for target in CATEGORY_TARGETS:
    temp = future_forecasts[future_forecasts["Target"] == target]
    fig.add_trace(go.Scatter(x=temp["Date"], y=temp["Forecast"], mode="lines+markers", name=target))
fig.update_layout(title="True 26 week future forecast by content category", xaxis_title="Future date", yaxis_title="Forecasted attention score", template="plotly_white", hovermode="x unified", height=650)
fig.show()

## Attention Intelligence Brief

This section converts the forecast outputs into a short strategy brief.

The rule based brief is the source of truth. The optional AI version only improves wording and does not change the values.

In [34]:
def create_rule_based_brief(platform_summary, category_summary, model_summary):
    top_platforms = platform_summary.sort_values("Expected percent change", ascending=False).reset_index(drop=True)
    top_categories = category_summary.sort_values("Expected percent change", ascending=False).reset_index(drop=True)
    best_models_text = model_summary.sort_values("Average_MAE").groupby("Group").first().reset_index()
    lines = []
    lines.append("Attention Intelligence Brief")
    lines.append("")
    lines.append("Main platform signal")
    lines.append(f"{top_platforms.iloc[0]['Target']} shows the strongest platform growth with an expected change of {top_platforms.iloc[0]['Expected percent change']:.2f} percent over the next 26 weeks.")
    lines.append("")
    lines.append("Main content signal")
    lines.append(f"{top_categories.iloc[0]['Target']} shows the strongest content category growth with an expected change of {top_categories.iloc[0]['Expected percent change']:.2f} percent over the next 26 weeks.")
    lines.append("")
    lines.append("Platform ranking")
    for i, row in top_platforms.iterrows():
        lines.append(f"{i + 1}. {row['Target']} with {row['Expected percent change']:.2f} percent expected change")
    lines.append("")
    lines.append("Content category ranking")
    for i, row in top_categories.iterrows():
        lines.append(f"{i + 1}. {row['Target']} with {row['Expected percent change']:.2f} percent expected change")
    lines.append("")
    lines.append("Model reliability")
    for _, row in best_models_text.iterrows():
        lines.append(f"For {row['Group'].lower()} targets, {row['Model']} had the best average backtest MAE.")
    lines.append("")
    lines.append("Business meaning")
    lines.append("The forecast can help a content or marketing team decide which platforms and content themes deserve closer monitoring during planning.")
    lines.append("")
    lines.append("Caution")
    lines.append("Google Trends, Wikipedia, and Reddit engagement are proxy signals. They support decision making, but they do not guarantee future behavior.")
    return "\n".join(lines)

rule_based_brief = create_rule_based_brief(platform_future_summary, category_future_summary, model_summary)
print(rule_based_brief)

Attention Intelligence Brief

Main platform signal
Reddit shows the strongest platform growth with an expected change of 22.63 percent over the next 26 weeks.

Main content signal
Entertainment and Media shows the strongest content category growth with an expected change of 110.66 percent over the next 26 weeks.

Platform ranking
1. Reddit with 22.63 percent expected change
2. LinkedIn with 17.32 percent expected change
3. TikTok with 2.12 percent expected change
4. Instagram with 1.97 percent expected change
5. YouTube with -1.22 percent expected change

Content category ranking
1. Entertainment and Media with 110.66 percent expected change
2. Artificial Intelligence and Technology with 86.45 percent expected change
3. Science and Education with 36.24 percent expected change
4. Mental Health and Self Improvement with 35.15 percent expected change
5. Culture Food and Travel with 31.84 percent expected change
6. Sports with 15.99 percent expected change
7. Gaming with 8.33 percent expec

In [35]:
def create_groq_brief(rule_based_brief):
    if not GROQ_AVAILABLE or not USE_GROQ:
        return rule_based_brief
    api_key = GROQ_API_KEY
    if not api_key:
        try:
            from getpass import getpass
            api_key = getpass("Enter Groq API key or press Enter to skip")
        except Exception:
            api_key = ""
    if not api_key:
        return rule_based_brief
    try:
        client = Groq(api_key=api_key)
        prompt = f"""
Rewrite the following Attention Intelligence Brief in simple academic English.
Do not change any number.
Do not change the ranking.
Do not add new facts.
Keep the same meaning.

Brief
{rule_based_brief}
"""
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        print("Groq brief failed. Rule based brief is used")
        print(e)
        return rule_based_brief

attention_brief = create_groq_brief(rule_based_brief)
print(attention_brief)

Enter Groq API key or press Enter to skip··········
Attention Intelligence Brief

Key Platform Indicator
Reddit is expected to show the most significant growth among platforms, with a predicted increase of 22.63 percent over the next 26 weeks.

Key Content Indicator
The Entertainment and Media category is expected to experience the most substantial growth, with a predicted increase of 110.66 percent over the next 26 weeks.

Platform Ranking
1. Reddit, with an expected change of 22.63 percent
2. LinkedIn, with an expected change of 17.32 percent
3. TikTok, with an expected change of 2.12 percent
4. Instagram, with an expected change of 1.97 percent
5. YouTube, with an expected change of -1.22 percent

Content Category Ranking
1. Entertainment and Media, with an expected change of 110.66 percent
2. Artificial Intelligence and Technology, with an expected change of 86.45 percent
3. Science and Education, with an expected change of 36.24 percent
4. Mental Health and Self Improvement, with 

## Gradio dashboard

The dashboard shows the true future forecast for any platform or content category.

In [36]:
import gradio as gr


def dashboard(target):
    history = model_data[target]
    forecast_df = future_forecasts[future_forecasts["Target"] == target]
    summary_row = future_summary[future_summary["Target"] == target].iloc[0]
    model_rows = results_df[results_df["Target"] == target].sort_values("MAE")
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=history.index, y=history.values, mode="lines", name="Historical"))
    fig.add_trace(go.Scatter(x=forecast_df["Date"], y=forecast_df["Forecast"], mode="lines+markers", name="True future forecast"))
    fig.update_layout(title=f"True 26 week future forecast for {target}", xaxis_title="Date", yaxis_title="Attention score", template="plotly_white", hovermode="x unified", height=500)
    interpretation = f"{target} is forecasted using {summary_row['Selected model']}. The expected change over the next 26 weeks is {summary_row['Expected percent change']:.2f} percent. The result is based on proxy attention signals."
    table = model_rows[["Model", "MAE", "RMSE", "MAPE"]].round(3)
    return fig, interpretation, table

with gr.Blocks(title="Human Attention Forecasting Dashboard") as demo:
    gr.Markdown("# Human Attention Forecasting Dashboard")
    gr.Markdown("Select a platform or content category to view the historical signal, future forecast, and model comparison.")
    target_input = gr.Dropdown(choices=ALL_TARGETS, value="YouTube", label="Target")
    plot_output = gr.Plot(label="Historical trend and future forecast")
    text_output = gr.Textbox(label="Interpretation")
    table_output = gr.Dataframe(label="Backtest model comparison")
    target_input.change(fn=dashboard, inputs=target_input, outputs=[plot_output, text_output, table_output])
    demo.load(fn=dashboard, inputs=target_input, outputs=[plot_output, text_output, table_output])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5f54747e75bb70ac52.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Export outputs

This section saves the final outputs for backup and for Streamlit deployment.

In [37]:
output_dir = Path("attention_forecasting_outputs")
output_dir.mkdir(exist_ok=True)

platform_trends.to_csv(output_dir / "platform_trends.csv")
category_trends.to_csv(output_dir / "google_category_trends.csv")
category_attention.to_csv(output_dir / "category_attention_index.csv")
wiki_data.to_csv(output_dir / "wikipedia_pageviews.csv")
reddit_weekly.to_csv(output_dir / "reddit_weekly_category_engagement.csv")
reddit_category_summary.to_csv(output_dir / "reddit_category_summary.csv", index=False)
model_data.to_csv(output_dir / "modeling_attention_data.csv")
results_df.to_csv(output_dir / "backtest_model_results.csv", index=False)
model_summary.to_csv(output_dir / "model_summary.csv", index=False)
best_models.to_csv(output_dir / "best_models.csv", index=False)
future_forecasts.to_csv(output_dir / "future_forecasts.csv", index=False)
future_summary.to_csv(output_dir / "future_summary.csv", index=False)
platform_future_summary.to_csv(output_dir / "platform_future_summary.csv", index=False)
category_future_summary.to_csv(output_dir / "category_future_summary.csv", index=False)
source_status_df.to_csv(output_dir / "source_status.csv", index=False)

with open(output_dir / "attention_intelligence_brief.txt", "w", encoding="utf-8") as f:
    f.write(attention_brief)

print("Exported files")
print(list(output_dir.iterdir()))

Exported files
[PosixPath('attention_forecasting_outputs/backtest_model_results.csv'), PosixPath('attention_forecasting_outputs/platform_future_summary.csv'), PosixPath('attention_forecasting_outputs/best_models.csv'), PosixPath('attention_forecasting_outputs/platform_trends.csv'), PosixPath('attention_forecasting_outputs/reddit_weekly_category_engagement.csv'), PosixPath('attention_forecasting_outputs/attention_intelligence_brief.txt'), PosixPath('attention_forecasting_outputs/future_summary.csv'), PosixPath('attention_forecasting_outputs/modeling_attention_data.csv'), PosixPath('attention_forecasting_outputs/future_forecasts.csv'), PosixPath('attention_forecasting_outputs/category_attention_index.csv'), PosixPath('attention_forecasting_outputs/reddit_category_summary.csv'), PosixPath('attention_forecasting_outputs/source_status.csv'), PosixPath('attention_forecasting_outputs/wikipedia_pageviews.csv'), PosixPath('attention_forecasting_outputs/model_summary.csv'), PosixPath('attention_

In [38]:
!zip -r attention_forecasting_outputs.zip attention_forecasting_outputs

  adding: attention_forecasting_outputs/ (stored 0%)
  adding: attention_forecasting_outputs/backtest_model_results.csv (deflated 59%)
  adding: attention_forecasting_outputs/platform_future_summary.csv (deflated 37%)
  adding: attention_forecasting_outputs/best_models.csv (deflated 45%)
  adding: attention_forecasting_outputs/platform_trends.csv (deflated 73%)
  adding: attention_forecasting_outputs/reddit_weekly_category_engagement.csv (deflated 53%)
  adding: attention_forecasting_outputs/attention_intelligence_brief.txt (deflated 59%)
  adding: attention_forecasting_outputs/future_summary.csv (deflated 46%)
  adding: attention_forecasting_outputs/modeling_attention_data.csv (deflated 80%)
  adding: attention_forecasting_outputs/future_forecasts.csv (deflated 82%)
  adding: attention_forecasting_outputs/category_attention_index.csv (deflated 83%)
  adding: attention_forecasting_outputs/reddit_category_summary.csv (deflated 36%)
  adding: attention_forecasting_outputs/source_status.c

## Streamlit deployment

The deployed app uses the exported output files from this notebook. It does not retrain the models.

https://human-attention-forecasting.streamlit.app/

## Conclusion and limitations

This project delivers the approved scope. It forecasts attention for platforms and content categories using Google Trends, Wikipedia page views, and Reddit Kaggle engagement data.

The models are evaluated through a 26 week backtest. The final output is a true 26 week future forecast.

The main limitation is that all three sources are proxy signals. They do not measure total human attention directly. The forecasts should be used as decision support, not as guaranteed predictions.